# Шаг 02. Проверить разметку 100 фотографий

`annotations/worker_reference_first_pass.json` содержит первый визуальный проход по 100 отдельным CC0-фотографиям. Для каждой фотографии есть `photo_id`, ссылка на оригинал и нормализованные рамки взрослых бабочек. `boxes: []` означает, что взрослых бабочек на снимке нет.

Это **черновик ответов для обучения, экзаменов и контроля**. Перед использованием в экзамене или контрольных заданиях второй участник команды должен независимо просмотреть фотографии и подтвердить или исправить ответы. Приоритетные случаи перечислены в `annotations/worker_reference_review_notes.json`. Два снимка с бабочкой только на дисплее пока не следует использовать в экзамене и контроле.

Пулы этот ноутбук не создаёт и не запускает. Роли фотографий пока не назначаются: сначала нужно подтвердить ответы.

In [ ]:
from pathlib import Path
import json
from collections import Counter

ROOT = Path.cwd()
RUN = "20261108T001420_435829Z"
MANIFEST = ROOT / "sampling_runs" / RUN / "worker_training_exam_draft.jsonl"
ANNOTATIONS = ROOT / "annotations" / "worker_reference_first_pass.json"
REVIEW_NOTES = ROOT / "annotations" / "worker_reference_review_notes.json"
CONTEST = ROOT / "dataset.jsonl"
MAIN = ROOT / "sampling_runs" / RUN / "main_dataset_draft.jsonl"

read_jsonl = lambda path: [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
manifest = read_jsonl(MANIFEST)
annotations = json.loads(ANNOTATIONS.read_text(encoding="utf-8"))
review_notes = json.loads(REVIEW_NOTES.read_text(encoding="utf-8"))
contest = read_jsonl(CONTEST)
main = read_jsonl(MAIN)
print("Фотографий:", len(annotations), "Приоритетных для проверки:", len(review_notes["priority_review"]))

In [ ]:
import math

assert len(manifest) == len(annotations) == 100
manifest_ids = [row["photo_id"] for row in manifest]
annotation_ids = [row["photo_id"] for row in annotations]
assert len(set(manifest_ids)) == len(set(annotation_ids)) == 100
assert set(manifest_ids) == set(annotation_ids)
assert not set(annotation_ids) & {row["photo_id"] for row in contest}
assert not set(annotation_ids) & {row["photo_id"] for row in main}
assert not {row["photo_url"] for row in annotations} & {row["photo_url"] for row in contest}
assert not {row["photo_url"] for row in annotations} & {row["photo_url"] for row in main}
manifest_by_id = {row["photo_id"]: row for row in manifest}
for row in annotations:
    assert set(row) == {"photo_id", "photo_url", "boxes"}
    assert row["photo_url"] == manifest_by_id[row["photo_id"]]["photo_url"]
    assert isinstance(row["boxes"], list)
    for box in row["boxes"]:
        assert set(box) == {"left", "top", "width", "height"}
        assert all(isinstance(box[key], (int, float)) and math.isfinite(box[key]) for key in box)
        assert 0 <= box["left"] < 1 and 0 <= box["top"] < 1
        assert 0 < box["width"] <= 1 and 0 < box["height"] <= 1
        assert box["left"] + box["width"] <= 1
        assert box["top"] + box["height"] <= 1
print("Проверки пройдены. С бабочками:", sum(bool(row["boxes"]) for row in annotations),
      "без бабочек:", sum(not row["boxes"] for row in annotations),
      "рамок:", sum(len(row["boxes"]) for row in annotations))

## Визуальная проверка

Вызовите `show_examples(start=0, count=12)` и пролистайте все 100 снимков партиями. Красные рамки — предложенная разметка; под снимком есть ссылка на оригинал. Сверяйте по оригиналу мелкие объекты и усики. Поправки вносите в `annotations/worker_reference_first_pass.json`, после чего повторно выполните проверки выше и обновите одноимённый JSONL. Для двух фотографий с бабочкой только на дисплее сначала согласуйте правило команды.

In [ ]:
from html import escape
from IPython.display import HTML, display

def show_examples(start=0, count=12):
    if not (0 <= start < len(annotations)):
        raise ValueError("start вне диапазона")
    cards = []
    for row in annotations[start:start + count]:
        photo_id = row["photo_id"]
        url = escape(row["photo_url"], quote=True)
        overlays = "".join(
            f'<div style="position:absolute;left:{100*b["left"]}%;top:{100*b["top"]}%;'
            f'width:{100*b["width"]}%;height:{100*b["height"]}%;'
            'border:2px solid #e11;box-sizing:border-box;pointer-events:none"></div>'
            for b in row["boxes"]
        )
        cards.append(
            f'<div style="width:360px;min-width:360px">'
            f'<div style="position:relative;display:block"><img src="{url}" '
            'style="width:100%;display:block"/>' + overlays + '</div>'
            f'<div><b>{photo_id}</b> · рамок: {len(row["boxes"])} · '
            f'<a href="{url}" target="_blank">оригинал</a></div></div>'
        )
    display(HTML('<div style="display:flex;flex-wrap:wrap;gap:16px">' + ''.join(cards) + '</div>'))

show_examples(start=0, count=12)

После независимой проверки можно распределить подтверждённые фотографии между обучением, экзаменом и контролем. Не используйте спорные кадры как экзаменационные или контрольные до согласованного решения команды.